# Notebook 09 — Baseline Models

## Purpose
I train and evaluate simple baseline models before any advanced methods.

## Why this matters
Baselines define the floor of performance.  If an advanced model cannot
significantly beat a majority-class dummy, it adds no value.

## Models
1. Dummy classifier (majority class) — the absolute floor
2. Logistic Regression — simple linear baseline

## Tasks
- Task A: Binary classification — is_late (will delivery be late?)
- Task B: 5-class classification — review_score (1–5 stars)

## Outputs
`models/baseline_results.json`


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib

sys.path.insert(0, str(Path('..').resolve()))
from src.config import load_config
from src.paths import Paths
from src.metrics import bootstrap_metric, permutation_test, macro_f1, model_summary_table
from src.utils import set_all_seeds, save_metrics

cfg   = load_config()
paths = Paths(cfg)
SEED  = cfg['project']['random_seed']
set_all_seeds(SEED)

train = pd.read_parquet(paths.processed / 'train.parquet')
test  = pd.read_parquet(paths.processed / 'test.parquet')
print(f"Train: {len(train):,}  Test: {len(test):,}")

Random seed set to 42
Train: 43,695  Test: 52,783


In [3]:
# Prepare feature matrices
FEAT_REVIEW = cfg['modeling']['feature_cols_review']
FEAT_LATE   = cfg['modeling']['feature_cols_late']

# Task B: review score prediction
# Drop rows with missing review score
train_rv = train.dropna(subset=['review_score'] + FEAT_REVIEW)
test_rv  = test.dropna(subset=['review_score']  + FEAT_REVIEW)

X_train_rv = train_rv[FEAT_REVIEW].fillna(0).values
y_train_rv = train_rv['review_score'].astype(int).values
X_test_rv  = test_rv[FEAT_REVIEW].fillna(0).values
y_test_rv  = test_rv['review_score'].astype(int).values

print(f"Task B (review score) — Train: {len(X_train_rv):,}  Test: {len(X_test_rv):,}")
print(f"Class distribution in train: {dict(pd.Series(y_train_rv).value_counts().sort_index())}")

# Task A: late delivery prediction
train_lt = train.dropna(subset=['is_late'] + FEAT_LATE)
test_lt  = test.dropna(subset=['is_late']  + FEAT_LATE)

X_train_lt = train_lt[FEAT_LATE].fillna(0).values
y_train_lt = train_lt['is_late'].astype(int).values
X_test_lt  = test_lt[FEAT_LATE].fillna(0).values
y_test_lt  = test_lt['is_late'].astype(int).values

print(f"Task A (late delivery) — Train: {len(X_train_lt):,}  Test: {len(X_test_lt):,}")
print(f"Late delivery rate (train): {y_train_lt.mean()*100:.1f}%")


Task B (review score) — Train: 42,597  Test: 51,877
Class distribution in train: {1: np.int64(3823), 2: np.int64(1311), 3: np.int64(3647), 4: np.int64(8613), 5: np.int64(25203)}
Task A (late delivery) — Train: 43,693  Test: 52,777
Late delivery rate (train): 5.6%


In [5]:
# --- TASK B: REVIEW SCORE BASELINE MODELS ---
print("=" * 50)
print("TASK B: Review score prediction (1-5)")
print("=" * 50)

# Baseline 1: Dummy (most frequent)
dummy_rv = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy_rv.fit(X_train_rv, y_train_rv)

y_pred_dummy_rv = dummy_rv.predict(X_test_rv)

dummy_acc = accuracy_score(y_test_rv, y_pred_dummy_rv)
dummy_f1 = macro_f1(y_test_rv, y_pred_dummy_rv)

predicted_class = dummy_rv.classes_[np.argmax(dummy_rv.class_prior_)]

print("\nDummy (most frequent):")
print(f"  Accuracy: {dummy_acc:.4f}")
print(f"  Macro F1: {dummy_f1:.4f}")
print(f"  (always predicts: {predicted_class})")

TASK B: Review score prediction (1-5)

Dummy (most frequent):
  Accuracy: 0.5933
  Macro F1: 0.1489
  (always predicts: 5)


In [6]:
# Baseline 2: Logistic Regression
lr_pipeline_rv = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=SEED,
                               class_weight='balanced')),
])
lr_pipeline_rv.fit(X_train_rv, y_train_rv)
y_pred_lr_rv = lr_pipeline_rv.predict(X_test_rv)

lr_acc = accuracy_score(y_test_rv, y_pred_lr_rv)
lr_f1  = macro_f1(y_test_rv, y_pred_lr_rv)
print(f"Logistic Regression (Task B):")
print(f"  Accuracy: {lr_acc:.4f}")
print(f"  Macro F1: {lr_f1:.4f}")
print()
print(classification_report(y_test_rv, y_pred_lr_rv, zero_division=0))


Logistic Regression (Task B):
  Accuracy: 0.3065
  Macro F1: 0.2240

              precision    recall  f1-score   support

           1       0.28      0.53      0.37      5344
           2       0.03      0.16      0.05      1566
           3       0.08      0.26      0.12      4184
           4       0.21      0.08      0.11     10004
           5       0.68      0.36      0.47     30779

    accuracy                           0.31     51877
   macro avg       0.26      0.28      0.22     51877
weighted avg       0.48      0.31      0.35     51877



In [7]:
# Bootstrap CIs
print("=== Bootstrap 95% CIs (Task B) ===")
n_boot = cfg['modeling']['n_bootstrap']

dummy_ci = bootstrap_metric(y_test_rv, y_pred_dummy_rv, n_bootstrap=n_boot, seed=SEED)
lr_ci    = bootstrap_metric(y_test_rv, y_pred_lr_rv,    n_bootstrap=n_boot, seed=SEED)

print(f"Dummy  : {dummy_ci['point']:.4f}  [{dummy_ci['lower']:.4f}, {dummy_ci['upper']:.4f}]")
print(f"LogReg : {lr_ci['point']:.4f}  [{lr_ci['lower']:.4f}, {lr_ci['upper']:.4f}]")


=== Bootstrap 95% CIs (Task B) ===
Dummy  : 0.5933  [0.5889, 0.5975]
LogReg : 0.3065  [0.3029, 0.3103]


In [8]:
# Permutation test for LogReg
perm_rv = permutation_test(y_test_rv, y_pred_lr_rv,
                           n_permutations=cfg['modeling']['n_permutations'], seed=SEED)
print(f"\nPermutation test (LogReg, Task B):")
print(f"  Observed accuracy:  {perm_rv['observed']:.4f}")
print(f"  Null mean:          {perm_rv['null_mean']:.4f}")
print(f"  p-value:            {perm_rv['p_value']:.4f}")
sig = "SIGNIFICANT" if perm_rv['p_value'] < 0.05 else "NOT significant"
print(f"  Result: {sig} (alpha=0.05)")



Permutation test (LogReg, Task B):
  Observed accuracy:  0.3065
  Null mean:          0.2454
  p-value:            0.0000
  Result: SIGNIFICANT (alpha=0.05)


In [9]:
# --- TASK A: LATE DELIVERY ---
print("=" * 50)
print("TASK A: Late delivery prediction (binary)")
print("=" * 50)

dummy_lt = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy_lt.fit(X_train_lt, y_train_lt)
y_pred_dummy_lt = dummy_lt.predict(X_test_lt)

lr_pipeline_lt = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=SEED)),
])
lr_pipeline_lt.fit(X_train_lt, y_train_lt)
y_pred_lr_lt = lr_pipeline_lt.predict(X_test_lt)

print(f"Dummy  accuracy (Task A): {accuracy_score(y_test_lt, y_pred_dummy_lt):.4f}")
print(f"LogReg accuracy (Task A): {accuracy_score(y_test_lt, y_pred_lr_lt):.4f}")
print()
print(classification_report(y_test_lt, y_pred_lr_lt, zero_division=0))


TASK A: Late delivery prediction (binary)
Dummy  accuracy (Task A): 0.9227
LogReg accuracy (Task A): 0.9227

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     48699
           1       0.00      0.00      0.00      4078

    accuracy                           0.92     52777
   macro avg       0.46      0.50      0.48     52777
weighted avg       0.85      0.92      0.89     52777



In [10]:
# Save baseline results
baseline_results = {
    'task_review_score': {
        'dummy':  {'accuracy': dummy_ci['point'], **dummy_ci},
        'logreg': {'accuracy': lr_ci['point'],    **lr_ci,
                   'permutation_p': perm_rv['p_value']},
    },
    'task_late_delivery': {
        'dummy':  {'accuracy': float(accuracy_score(y_test_lt, y_pred_dummy_lt))},
        'logreg': {'accuracy': float(accuracy_score(y_test_lt, y_pred_lr_lt))},
    }
}
save_metrics(baseline_results, paths.models / 'baseline_results.json')
print("Notebook 09 complete. Baseline results saved.")
print("Next: Notebook 10 — Advanced Models (RF + XGBoost + K-Means)")


  Saved metrics: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\models\baseline_results.json
Notebook 09 complete. Baseline results saved.
Next: Notebook 10 — Advanced Models (RF + XGBoost + K-Means)
